In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

count = CountVectorizer()
docs = np.array(['The sun is shining','The weather is sweet','The sun is shining, the weather is sweet, and one and one is two'])

In [ ]:
bag = count.fit_transform(docs)

In [ ]:
print(count.vocabulary_)

In [ ]:
print(bag.toarray())

In [ ]:
df = pd.read_csv("../src/movies_data.csv",encoding='utf-8')
df.sample(10)

In [ ]:
df.loc[1,'review']

## Preprocess the data (remove html tags, unwanted symbols)

In [ ]:
import re
def preprocessor(text):
    text = re.sub('<[^>]*>','',text)
    emoticons = re.findall('(?::|;|=)(?:-)?(?:\)|\(|D|P)',text)
    text = (re.sub('[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', ''))
    return text

In [ ]:
df['review'] = df['review'].apply(preprocessor)

In [ ]:
df.loc[1,'review']

## Tokenize the review column and remove stop words (not neccessary if using tf-idf)

In [ ]:
from nltk.stem.porter import PorterStemmer
porter = PorterStemmer()

def tokenizer_porter(text):
    return [porter.stem(word) for word in text.split()]

def tokenizer(text):
    return text.split()

In [ ]:
from nltk.corpus import stopwords
stop = stopwords.words('english')

## TF-IDF + Logistic Regression

In [ ]:
X_train = df.loc[:25000, 'review'].values
y_train = df.loc[:25000, 'sentiment'].values
X_test = df.loc[25000:, 'review'].values
y_test = df.loc[25000:, 'sentiment'].values

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(strip_accents=None,
                       lowercase=False,
                       preprocessor=None)

small_param_grid = [
    {
        'vect__ngram_range':[(1,1)],
        'vect__stop_words':[None],
        'vect__tokenizer':[tokenizer,tokenizer_porter],
        'clf__penalty':['l2','l1'],
        'clf__C':[1.0,10.0]
    },
    {
        'vect__ngram_range':[(1,1)],
        'vect__stop_words':[stop,None],
        'vect__tokenizer':[tokenizer],
        'vect__use_idf':[False],
        'vect__norm':[None],
        'clf__penalty':['l2','l1'],
        'clf__C':[1.0,10.0]
    }
]

lr_tfidf = Pipeline(
    [
        ('vect',tfidf),
        ('clf', LogisticRegression(solver='liblinear'))
    ]
)

gs_lr_tfidf = GridSearchCV(lr_tfidf, small_param_grid, cv=5, verbose=2, n_jobs=-1, scoring='accuracy')
gs_lr_tfidf.fit(X_train,y_train)

In [ ]:
gs_lr_tfidf.best_params_

In [ ]:
print(f"CV Accuracy: {gs_lr_tfidf.best_score_:.3f}")
clf = gs_lr_tfidf.best_estimator_
print(f"Test Accuracy: {clf.score(X_test,y_test):.3f}")

In [ ]:
texts = ["The movie was fantastic, i loved the characters in the film, though i felt it was a bit slow","The movie was incredibly mediocre, i mean just the fact that it was 3 hours long, made it unbearable","The movie was alright. It could had more action scenes but the fight scenes were good"]

predictions = clf.predict(texts)

for text, pred in zip(texts, predictions):
    print(f"{text} --> {pred}")

## Save the model

In [ ]:
from joblib import dump, load

dump(clf, "model/sentiment_model.joblib")